g# Reliable and Adaptive Agentic RAG System (Step 3)

This notebook extends the Step 2 multi-agent retrieval system with reliability, adaptation, recovery, and trust mechanisms.

The goal is to make the RAG system more reliable when evidence is:

weak
incomplete
ambiguous
contradictory

The notebook implements multiple reliability mechanisms and adaptive orchestration behaviors.

## 1. Installation

In [2]:
!pip install -q pandas numpy pytrec_eval transformers accelerate

  error: subprocess-exited-with-error
  
  × Building wheel for pytrec_eval (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [41 lines of output]
      Fetching trec_eval from https://github.com/usnistgov/trec_eval/archive/v9.0.8.tar.gz.
      C:\Users\stude\AppData\Local\Temp\pip-build-env-sp1fvrnj\overlay\Lib\site-packages\setuptools\dist.py:599: SetuptoolsDeprecationWarning: Invalid dash-separated key 'description-file' in 'metadata' (setup.cfg), please use the underscore name 'description_file' instead.
      !!
      
              ********************************************************************************
              Usage of dash-separated 'description-file' will not be supported in future
              versions. Please use the underscore name 'description_file' instead.
              (Affected: pytrec_eval).
      
              Available configuration options are listed in:
              https://setuptools.pypa.io/en/latest/userguide/declarative_config.

## 2. Imports

In [3]:
import re
import time
import random
import numpy as np
import pandas as pd

from collections import defaultdict

## 3. Load Step 2 Components

This section imports the orchestration strategies and retrievers
from the Step 2 notebook implementation.

In [4]:
# Assumes Step 2 notebook has already been executed
# Required:
# - waterfall_orchestrate
# - voting_orchestrate
# - confidence_orchestrate
# - qa_data

## 4. Utility Functions

In [5]:
def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return set(text.split())



def overlap_score(a, b):

    ta = normalize(a)
    tb = normalize(b)

    if len(ta) == 0:
        return 0.0

    return len(ta & tb) / len(ta)

## 5. Mechanisms

### 5.1 Evidence Sufficiency Estimation (A)
This module estimates whether the retrieved evidence is sufficient.

Signals used:

- overlap between query and retrieved chunks
- number of supporting chunks
- retrieval fusion score

In [6]:
class EvidenceSufficiencyAgent:

    def assess(self, query, docs):

        if len(docs) == 0:
            return {
                "sufficient": False,
                "score": 0.0,
            }

        q_tokens = normalize(query)

        support_scores = []

        for d in docs[:5]:

            txt = d.page_content
            d_tokens = normalize(txt)

            overlap = len(q_tokens & d_tokens)
            overlap = overlap / max(len(q_tokens), 1)

            support_scores.append(overlap)

        avg_support = np.mean(support_scores)

        sufficient = avg_support >= 0.15

        return {
            "sufficient": sufficient,
            "score": round(float(avg_support), 3),
            "supporting_chunks": len([
                x for x in support_scores if x > 0.1
            ])
        }

### %.2 Groundness / Support Verification (B)
The answer is verified against retrieved evidence.

In [7]:
class GroundednessAgent:

    def verify(self, answer, docs):

        ans_tokens = normalize(answer)

        if len(ans_tokens) == 0:
            return False

        for d in docs:

            txt = d.page_content
            doc_tokens = normalize(txt)

            overlap = len(ans_tokens & doc_tokens)

            if overlap >= max(1, int(len(ans_tokens) * 0.2)):
                return True

        return False

### 5.3 Contradiction Detection (C)
This mechanism checks whether retrieved chunks contain conflicting statements.

Lightweight heuristic:

- detect conflicting keywords
- detect opposite numeric claims

In [8]:
class ContradictionAgent:

    CONTRADICTIONS = [
        ("yes", "no"),
        ("increase", "decrease"),
        ("true", "false"),
    ]

    def detect(self, docs):

        texts = [d.page_content.lower() for d in docs[:5]]

        for a, b in self.CONTRADICTIONS:

            has_a = any(a in t for t in texts)
            has_b = any(b in t for t in texts)

            if has_a and has_b:
                return {
                    "contradiction": True,
                    "reason": f"Detected conflict: {a} vs {b}"
                }

        return {
            "contradiction": False,
            "reason": "No obvious contradiction detected"
        }

### 5.4 Clarification Strategy (D)
The system detects ambiguous or underspecified questions.

In [9]:
class ClarificationAgent:

    def needs_clarification(self, query):

        q = query.lower().strip()

        ambiguous = [
            "it",
            "they",
            "this",
            "that",
        ]

        short_query = len(q.split()) <= 3

        ambiguous_ref = any(x in q.split() for x in ambiguous)

        if short_query or ambiguous_ref:
            return True

        return False

    def clarification_question(self, query):

        return (
            "Could you clarify your question or provide "
            "more specific details?"
        )

### 5.5 Abstention Mechanism (E)
The system abstains when evidence is weak or contradictory.

In [10]:
class AbstentionAgent:

    def should_abstain(self,
                       sufficiency,
                       grounded,
                       contradiction):

        if not sufficiency["sufficient"]:
            return True

        if contradiction["contradiction"]:
            return True

        if not grounded:
            return True

        return False

### 5.6 Self-Reflection / Critique Loop (F)
The critic reviews the draft answer.

In [11]:
class CriticAgent:

    def critique(self,
                 answer,
                 grounded,
                 contradiction):

        feedback = []

        if not grounded:
            feedback.append("Answer weakly supported")

        if contradiction["contradiction"]:
            feedback.append("Evidence conflict detected")

        if len(answer.split()) < 3:
            feedback.append("Answer too short")

        if len(feedback) == 0:
            feedback.append("Answer appears acceptable")

        return feedback

### 5.7 Recovery Mechanism (G)
The system changes behavior dynamically.

Recovery actions:

- switch retrieval strategy
- rewrite query
- move to clarification mode
- abstain

In [12]:
class RecoveryAgent:

    def rewrite_query(self, query):

        return query + " ETH Zurich"

    def recover(self,
                query,
                current_strategy,
                sufficiency,
                contradiction):

        if contradiction["contradiction"]:

            return {
                "action": "switch_strategy",
                "new_strategy": "voting"
            }

        if not sufficiency["sufficient"]:

            rewritten = self.rewrite_query(query)

            return {
                "action": "rewrite_query",
                "query": rewritten
            }

        return {
            "action": "none"
        }

### 5.8 Trust / Confidence Scoring (H)
The confidence score combines:

- evidence sufficiency
- groundedness
- contradiction detection

In [14]:
class TrustAgent:

    def score(self,
              sufficiency,
              grounded,
              contradiction):

        score = 0.0

        score += sufficiency["score"] * 0.6

        if grounded:
            score += 0.3

        if contradiction["contradiction"]:
            score -= 0.4

        score = max(0.0, min(1.0, score))

        if score > 0.7:
            label = "HIGH"
        elif score > 0.4:
            label = "MEDIUM"
        else:
            label = "LOW"

        return {
            "score": round(float(score), 3),
            "label": label
        }

## 6. Adaptive Reliable RAG Orchestrator
This orchestrator integrates all reliability mechanisms.

In [15]:
class ReliableAdaptiveRAG:

    def __init__(self):

        self.sufficiency = EvidenceSufficiencyAgent()
        self.groundedness = GroundednessAgent()
        self.contradiction = ContradictionAgent()
        self.clarification = ClarificationAgent()
        self.abstention = AbstentionAgent()
        self.critic = CriticAgent()
        self.recovery = RecoveryAgent()
        self.trust = TrustAgent()

    def retrieve(self,
                 query,
                 strategy="confidence",
                 top_k=5):

        if strategy == "waterfall":
            docs, trace = waterfall_orchestrate(query, top_k)

        elif strategy == "voting":
            docs, trace = voting_orchestrate(query, top_k)

        else:
            docs, trace = confidence_orchestrate(query, top_k)

        return docs, trace

    def generate_answer(self, docs):

        if len(docs) == 0:
            return "NOT FOUND"

        return docs[0].page_content[:250]

    def run(self,
            query,
            strategy="confidence"):

        trace_log = []

        if self.clarification.needs_clarification(query):

            trace_log.append("Clarification triggered")

            return {
                "mode": "clarification",
                "response": self.clarification.clarification_question(query),
                "trace": trace_log
            }

        docs, trace = self.retrieve(query, strategy)

        trace_log.extend(trace)

        answer = self.generate_answer(docs)

        suff = self.sufficiency.assess(query, docs)

        grounded = self.groundedness.verify(answer, docs)

        contradiction = self.contradiction.detect(docs)

        trust = self.trust.score(
            suff,
            grounded,
            contradiction,
        )

        critique = self.critic.critique(
            answer,
            grounded,
            contradiction,
        )

        abstain = self.abstention.should_abstain(
            suff,
            grounded,
            contradiction,
        )

        recovery = self.recovery.recover(
            query,
            strategy,
            suff,
            contradiction,
        )

        trace_log.append(f"Trust score: {trust['score']}")
        trace_log.append(f"Recovery action: {recovery['action']}")

        if abstain:

            trace_log.append("System abstained")

            return {
                "mode": "abstain",
                "response": "The system cannot answer reliably.",
                "trust": trust,
                "critique": critique,
                "recovery": recovery,
                "trace": trace_log,
            }

        trace_log.append("Answer generated successfully")

        return {
            "mode": "answer",
            "response": answer,
            "trust": trust,
            "critique": critique,
            "recovery": recovery,
            "trace": trace_log,
        }

## 7. Initialize System

In [16]:
rag_system = ReliableAdaptiveRAG()

## 8. Example Queries
This section demonstrates:

- clarification behavior
- abstention
- recovery
- contradiction handling
- successful answer generation

In [18]:
queries = [
    "Who received ERC grants at ETH?",
    "How does ETH support innovation?",
    "it",
    "What research areas are important at ETH Zurich?"
]

for q in queries:

    print("\n" + "=" * 80)
    print("QUERY:", q)

    result = rag_system.run(q)

    print("\nMODE:", result["mode"])
    print("RESPONSE:", result["response"])

    if "trust" in result:
        print("\nTRUST:", result["trust"])

    if "critique" in result:
        print("\nCRITIQUE:")
        for c in result["critique"]:
            print("-", c)

    if "recovery" in result:
        print("\nRECOVERY:", result["recovery"])

    print("\nTRACE:")
    for t in result["trace"]:
        print("-", t)


QUERY: Who received ERC grants at ETH?


NameError: name 'confidence_orchestrate' is not defined

## 9. Benchmark Evaluation
This section evaluates:

- reliability scores
- latency
- system behavior
- abstention frequency

In [19]:
results = []

N = min(20, len(qa_data))

for i in range(N):

    q = qa_data[i]["question"]

    start = time.time()

    result = rag_system.run(q)

    runtime = time.time() - start

    trust_score = (
        result["trust"]["score"]
        if "trust" in result
        else 0.0
    )

    results.append({
        "query": q,
        "mode": result["mode"],
        "trust": trust_score,
        "runtime": runtime,
    })

benchmark_df = pd.DataFrame(results)

benchmark_df.head()

NameError: name 'qa_data' is not defined

## 10. Aggregate Statistics


In [20]:
benchmark_df.groupby("mode")[["trust", "runtime"]].mean()

NameError: name 'benchmark_df' is not defined

In [21]:
benchmark_df["mode"].value_counts()

NameError: name 'benchmark_df' is not defined

## 11. Failure Analysis

This section examines low-confidence cases.

In [22]:
low_conf = benchmark_df[
    benchmark_df["trust"] < 0.4
]

low_conf

NameError: name 'benchmark_df' is not defined

## 12. Final Discussion

This notebook implemented a reliable and adaptive agentic RAG framework aligned with the official Step 3 requirements.

Implemented capabilities:

- evidence sufficiency estimation
- groundedness verification
- contradiction detection
- clarification handling
- abstention
- self-reflection
- adaptive recovery
- trust estimation

The system dynamically adapts its behavior when failures or uncertainty are detected.

Limitations:

- heuristic-based reliability estimation
- lightweight contradiction detection
- simple answer generation

Future improvements:

- stronger verifier models
- semantic contradiction detection
- reinforcement learning orchestration
- retrieval strategy optimization
- claim-level grounding verification